# Notebook 06 — VAE Anomaly Detection
**Syllabus Week 12 | CLO 6: Generative Models & Anomaly Detection**

## Problem Statement
Train a **Variational Autoencoder** on normal weather observations to detect
anomalous weather events via reconstruction error.

The key insight: train only on "normal" data. At inference, abnormal inputs
(e.g. extreme heat waves, unusual humidity spikes) produce high reconstruction
MSE — the VAE cannot encode and reconstruct patterns it has never seen.

**Input:** single hourly observation vector of 7 weather features (NOT a window).

| Hyperparameter | Value |
|---|---|
| Input dim | 7 (standardised features) |
| Encoder | Linear(7→32→16) → (μ, logσ²) |
| Latent dim | 8 |
| Decoder | Linear(8→16→32→7) |
| β schedule | 0.0 → 0.5 linear over 10 epochs |
| Total epochs | 15 |
| Threshold | 95th percentile of train recon MSE |
| Params | ~2.8K |

## Section 2 — Math Derivation

### Variational Autoencoder
A VAE learns a latent variable model $p_\theta(x|z)p(z)$ by maximising the
Evidence Lower BOund (ELBO):

$$\mathcal{L}_{\text{ELBO}} = \underbrace{\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]}_{\text{reconstruction}} - \underbrace{D_{KL}(q_\phi(z|x) \| p(z))}_{\text{regularisation}}$$

### Encoder Output: Mean & Log-Variance
$$q_\phi(z|x) = \mathcal{N}(\mu_\phi(x),\, \text{diag}(\exp(\sigma^2_\phi(x))))$$

Two linear layers share the encoder trunk but produce separate outputs:
- $\mu = W_\mu h + b_\mu$
- $\log \sigma^2 = W_{\log\sigma^2} h + b_{\log\sigma^2}$

### Reparameterisation Trick
Backpropagation cannot pass through a stochastic node. Reparameterise:
$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

At inference (eval mode), use $z = \mu$ (deterministic) for consistent anomaly scores.

### Closed-Form KL for Gaussians
$$D_{KL}(\mathcal{N}(\mu, \sigma^2) \| \mathcal{N}(0, I)) =
  \frac{1}{2}\sum_{j=1}^{d}(\mu_j^2 + \sigma_j^2 - \log \sigma_j^2 - 1)$$

In code: $-\frac{1}{2}\sum(1 + \log\sigma^2 - \mu^2 - e^{\log\sigma^2})$

### β-VAE (Annealed)
$$\mathcal{L}_{\beta} = \mathbb{E}[\|x - \hat{x}\|^2] + \beta(t) \cdot D_{KL}$$

β starts at 0 (pure AE) and linearly increases to 0.5 over 10 epochs.
This prevents posterior collapse in early training when the decoder hasn't
yet learned to use $z$.

$$\beta(t) = \min\left(0.5,\, \frac{0.5 \cdot t}{10}\right), \quad t \in [1, 15]$$

### Anomaly Detection Threshold
$$\text{threshold} = \text{percentile}_{95}\{\|x_i - \hat{x}_i\|^2 : x_i \in \mathcal{D}_{\text{train}}\}$$

At inference: $\text{anomaly} = \mathbf{1}[\|x - \hat{x}\|^2 > \text{threshold}]$

In [ ]:
# Section 3 — Data Loading
# TODO: run after python -m ml.data_pipeline
import sys; sys.path.insert(0, '..')
import torch
import numpy as np

# VAE uses single-hour observations, NOT 48h windows
# from ml.train.train_vae import load_vae_data
# train_X, val_X, test_X = load_vae_data()   # each: (N, 7) float32 tensors
# print(f'train: {train_X.shape}')   # (N_train, 7)
# print(f'val  : {val_X.shape}')     # (N_val, 7)
# print(f'test : {test_X.shape}')    # (N_test, 7)

# Verify standardisation (train split only):
# print(f'train mean: {train_X.mean(0)}')   # should be ~0
# print(f'train std : {train_X.std(0)}')    # should be ~1
# print(f'val mean  : {val_X.mean(0)}')     # may differ — that is fine
# Note: scaler fitted on train 2023 only, applied to val/test

In [ ]:
# Section 4 — Model Definition & Shape Trace

# from ml.train.train_vae import WeatherVAE
# model = WeatherVAE(input_dim=7, latent_dim=8)
# n = sum(p.numel() for p in model.parameters())
# print(f'Params: {n:,}')   # ~2,800

# Shape trace:
# x         : (B, 7)      input (standardised single-hour observation)
# h_enc     : (B, 32)     Linear(7->32) + ReLU
# h_enc2    : (B, 16)     Linear(32->16) + ReLU
# mu        : (B, 8)      Linear(16->8)
# log_var   : (B, 8)      Linear(16->8)
# z         : (B, 8)      reparameterise: mu + std * eps
# h_dec     : (B, 16)     Linear(8->16) + ReLU
# h_dec2    : (B, 32)     Linear(16->32) + ReLU
# x_hat     : (B, 7)      Linear(32->7)   reconstruction

# Verify forward pass:
# import torch
# x = torch.randn(4, 7)
# x_hat, mu, log_var = model(x)
# print(f'x_hat   : {x_hat.shape}')    # (4, 7)
# print(f'mu      : {mu.shape}')       # (4, 8)
# print(f'log_var : {log_var.shape}')  # (4, 8)

# Verify reparameterisation (stochastic in train, deterministic in eval):
# model.train(); z1 = model.reparameterise(mu, log_var)
# model.train(); z2 = model.reparameterise(mu, log_var)
# model.eval();  z3 = model.reparameterise(mu, log_var)   # == mu
# print('z1==z2 (should be False):', torch.allclose(z1, z2))
# print('z3==mu (should be True) :', torch.allclose(z3, mu))

In [ ]:
# Section 5 — Training Results
# Run: python -m ml.train.train_vae

# TODO: paste epoch log here
# epoch 01  recon=X.XXXX  kl=X.XXXX  beta=0.00  total=X.XXXX
# ...
# epoch 10  recon=X.XXXX  kl=X.XXXX  beta=0.50  total=X.XXXX
# ...
# epoch 15  recon=X.XXXX  kl=X.XXXX  beta=0.50  total=X.XXXX
# Anomaly threshold (95th pct): X.XXXX

from IPython.display import Image
# Image('../models/vae/train_curve.png')

## Section 6 — Architecture Diagram

```
Input x  (B, 7)   ← standardised 7-feature weather observation
      │
  ┌───┴──────────────────────────┐
  │  ENCODER                    │
  │  Linear(7 → 32) + ReLU      │
  │  Linear(32 → 16) + ReLU     │
  │         │                   │
  │  ┌──────┴──────┐            │
  │  Linear(16→8)  Linear(16→8) │
  │     μ  (B,8)   logσ² (B,8)  │
  └─────────────────────────────┘
         │              │
         └──── z = μ + σ·ε  (reparameterise)
                    │
  ┌─────────────────┴───────────┐
  │  DECODER                   │
  │  Linear(8 → 16) + ReLU     │
  │  Linear(16 → 32) + ReLU    │
  │  Linear(32 → 7)            │
  └─────────────────────────────┘
         │
      x̂  (B, 7)   ← reconstruction

LOSS = MSE(x, x̂)  +  β(t) · KL(q(z|x) ‖ N(0,I))
       ───────────    ──────────────────────────────
       reconstruction      β: 0.0 → 0.5 (10 epochs)

ANOMALY THRESHOLD = 95th pct of train recon MSE  →  anomaly_threshold.json
```

In [ ]:
# Section 7 — Evaluation: Loss Curves + Latent Space Scatter
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image

# 7a. Training curves (dual-axis: recon MSE + KL divergence)
# Image('../models/vae/train_curve.png')

# 7b. Latent space scatter (PCA via manual SVD — no sklearn)
# Image('../models/vae/latent_scatter.png')

# 7c. Reconstruction error distribution on test set
# import torch
# from ml.train.train_vae import WeatherVAE
# import json
# model = WeatherVAE()
# model.load_state_dict(torch.load('../models/vae/model.pt', map_location='cpu'))
# model.eval()
# with open('../models/vae/anomaly_threshold.json') as f:
#     thresh = json.load(f)['threshold']
# with torch.no_grad():
#     x_hat, _, _ = model(test_X)
#     recon_mse = ((test_X - x_hat)**2).mean(dim=1).numpy()
# plt.figure(figsize=(8, 4))
# plt.hist(recon_mse, bins=50, edgecolor='k', alpha=0.7)
# plt.axvline(thresh, color='red', linestyle='--', label=f'threshold={thresh:.4f}')
# anomaly_rate = (recon_mse > thresh).mean()
# plt.title(f'Test recon MSE — anomaly rate {anomaly_rate:.1%}')
# plt.xlabel('Reconstruction MSE'); plt.legend()
# plt.tight_layout(); plt.show()

# TODO: paste anomaly rate on test set

## Section 8 — Reflection & Trade-offs

### Results (fill in after training)
| Metric | Value |
|---|---|
| Params | ~2,800 |
| Final recon MSE (train) | TODO |
| Final KL (train) | TODO |
| Anomaly threshold (95th pct) | TODO |
| Test anomaly rate | TODO % (expect ~5%) |
| Train time | TODO min |

### Design Decisions
| Decision | Alternative | Reason |
|---|---|---|
| Single-hour input (7-vec) | Windowed input (48, 7) | Simpler reconstruction — window VAEs need convolution and are harder to threshold |
| β-annealing 0→0.5 | Fixed β=1 (standard VAE) | Prevents posterior collapse early when decoder is untrained |
| 95th pct threshold | 99th / fixed constant | 95th gives ~5% false-positive rate on in-distribution data |
| Deterministic z=μ at eval | Sample at eval | Consistent anomaly scores across multiple calls |
| Manual SVD PCA | sklearn PCA | No sklearn dependency in training; SVD is the underlying algorithm anyway |
| Full 15 epochs (no early stop) | Early stop on val loss | VAE loss curves fluctuate more than supervised; β annealing makes early stopping noisy |

### Trade-offs vs Supervised Anomaly Detection
| Aspect | VAE (unsupervised) | Supervised classifier |
|---|---|---|
| Labels needed | None — train on normal data only | Requires labelled anomalies |
| Anomaly types | Detects any deviation | Only detects trained classes |
| False positive control | Via percentile threshold | Via decision boundary |
| Interpretability | Latent space is inspectable | Black-box for most classifiers |

### Limitations
- 7-dim input is very low-dimensional — anomaly separation may be weak.
- Threshold was computed on training distribution; drift in future weather
  patterns may require recalibration.
- No temporal context — a sudden temperature spike within a 24h period
  may not look abnormal in isolation.

### Numbers to fill in
- Best val recon MSE: **TODO**
- Epoch β reaches 0.5: **Epoch 10** (by design)
- Threshold value: **TODO**
- Most anomalous city in test set: **TODO**